# 10. BERT vs TF-IDF 추론 비용 분석

**교수님 Q3**: *"BERT 추론 비용(레이턴시, 메모리)은 얼마나 되는가?"*

| 항목 | TF-IDF | BERT |
|------|--------|------|
| 쿼리 레이턴시 평균 | 13.60 ms | 50.28 ms |
| 쿼리 레이턴시 p95 | 16.27 ms | 93.25 ms |
| 인덱스 메모리 | 11.1 MB | 63.3 MB |
| 초기화 시간 | 754 ms | 2,473 ms |

→ **BERT는 TF-IDF보다 3.7배 느리고 5.7배 더 많은 메모리를 사용**  
→ 단, 현재 아키텍처(Batch Processing)에서는 corpus 임베딩이 사전 계산돼 있어  
&nbsp;&nbsp;&nbsp;실시간 부담은 **쿼리 1건 인코딩(약 50ms)** 뿐

In [ ]:
# run_cost_analysis.py 실행 결과 기반 시각화
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

try:
    plt.rcParams['font.family'] = 'AppleGothic'
except:
    pass
plt.rcParams['axes.unicode_minus'] = False

# 측정값 (run_cost_analysis.py 결과)
results = {
    'TF-IDF': {'avg_ms': 13.60, 'p95_ms': 16.27, 'mem_mb': 11.1, 'init_ms': 754},
    'BERT':   {'avg_ms': 50.28, 'p95_ms': 93.25, 'mem_mb': 63.3, 'init_ms': 2473},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#4C9BE8', '#E85C5C']
models = ['TF-IDF', 'BERT']

# 레이턴시 비교
avg_vals = [results[m]['avg_ms'] for m in models]
p95_vals = [results[m]['p95_ms'] for m in models]
x = np.arange(2)
w = 0.35
axes[0].bar(x - w/2, avg_vals, w, label='평균', color=colors, alpha=0.85)
axes[0].bar(x + w/2, p95_vals, w, label='p95', color=colors, alpha=0.5, hatch='//')
for i, (a, p) in enumerate(zip(avg_vals, p95_vals)):
    axes[0].text(i - w/2, a + 1, f'{a:.1f}ms', ha='center', fontsize=10, fontweight='bold')
    axes[0].text(i + w/2, p + 1, f'{p:.1f}ms', ha='center', fontsize=10)
axes[0].set_xticks(x); axes[0].set_xticklabels(models)
axes[0].set_title('쿼리 레이턴시 (ms)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('응답 시간 (ms)')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# 메모리 비교
mem_vals = [results[m]['mem_mb'] for m in models]
bars = axes[1].bar(models, mem_vals, color=colors, alpha=0.85)
for bar, v in zip(bars, mem_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 1,
                 f'{v:.1f} MB', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('인덱스 메모리 (MB)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('메모리 (MB)')
axes[1].spines[['top','right']].set_visible(False)

# 초기화 시간
init_vals = [results[m]['init_ms'] for m in models]
bars = axes[2].bar(models, init_vals, color=colors, alpha=0.85)
for bar, v in zip(bars, init_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + 30,
                 f'{v:,.0f} ms', ha='center', fontsize=11, fontweight='bold')
axes[2].set_title('초기화 시간 (ms)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('시간 (ms)')
axes[2].spines[['top','right']].set_visible(False)

plt.suptitle('TF-IDF vs BERT 추론 비용 비교 (n=19,205건 corpus)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_figures/q3_cost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료: data/processed/eda_figures/q3_cost_analysis.png')

## 결론 — 언제 어떤 모델을 써야 하는가?

| 상황 | 권장 모델 | 이유 |
|------|---------|------|
| 실시간 응답 필요 (< 20ms) | **TF-IDF** | 평균 13.6ms, p95 16.3ms |
| 의미 기반 정확도 중요 | **BERT** | Hit@1 우위, 구어체 표현 처리 강점 |
| 메모리 제한 환경 | **TF-IDF** | 11MB vs 63MB |
| 자견 쿼리 (구어체 많음) | **BERT** | 구어체 의미 임베딩 강점 |
| 현재 배포 환경 (S3 정적) | **BERT** | Batch Pre-computation으로 비용 없음 |

**현재 시스템**: 임베딩 사전 계산 → S3 업로드 → 실시간 BERT 추론 없음  
→ 실질적 비용은 **쿼리 인코딩 50ms** 뿐 (RAG 파이프라인 기준)